# Example: escaping local optima with a multistart

Six normal modes at uneven distances. EM from a single start often ends in a local optimum: two components share
one mode and one component covers two. This notebook compares three strategies with the same budget of 300 EM
iterations: one long run, a multistart by the start, and emEM — ten short runs and a long one from the best.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

from pysatl_mpest.initializers import ClusterInitializer, MultiStartInitializer, RandomCentersPartitioner, RefiningInitializer

## Data

In [ ]:
means = [-20, -14, -4, 0, 9, 20]
truth = MixtureModel([normal(mu=m, sigma=1) for m in means])
X = truth.generate(size=900, random_state=3)
families = [normal] * len(means)
initializer = ClusterInitializer(RandomCentersPartitioner())

fig, ax = plt.subplots(figsize=(8, 3.5))
plot_mixtures(ax, X, {"true mixture": truth}, bins=100)
plt.show()

## Single runs

Ten single starts with independent generators, each followed by 300 EM iterations.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    singles = [em(300).fit(X, initializer.initialize(X, families, random_state=child)) for child in np.random.default_rng(0).spawn(10)]
single_scores = [mixture.loglikelihood(X) for mixture in singles]
print("single runs:", np.round(single_scores, 1))
print("true mixture:", round(truth.loglikelihood(X), 1))

## Three strategies

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    by_start = em(300).fit(X, MultiStartInitializer(initializer, n_init=10).initialize(X, families, random_state=0))
    short = RefiningInitializer(initializer, em(20))
    em_em = em(100).fit(X, MultiStartInitializer(short, n_init=10).initialize(X, families, random_state=0))

strategies = {"one run": singles[0], "multistart by the start": by_start, "emEM": em_em}
for name, mixture in strategies.items():
    print(f"{name:25} {mixture.loglikelihood(X):.1f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 3.5), sharey=True)
for ax, (name, mixture) in zip(axes, strategies.items()):
    plot_mixtures(ax, X, {name: mixture}, bins=100)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(sorted(single_scores), "o", label="single runs, sorted")
ax.axhline(em_em.loglikelihood(X), color="tab:red", label="emEM")
ax.axhline(truth.loglikelihood(X), color="black", linestyle="--", label="true mixture")
ax.set_ylabel("log-likelihood")
ax.legend()
plt.show()

assert em_em.loglikelihood(X) >= np.median(single_scores)

## Takeaway

The likelihood of a start says little about where EM will end: a multistart by the start chooses among starts, while
emEM chooses among short fits and is usually the better use of the same budget. Both are one line around an
existing initializer.